In [ ]:
import pandas as pd
import numpy as np

from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
session.file.get(
    "@MINOR_PROJECT.RAW.MY_CSV_STAGE/customer_shopping_behavior.csv",
    "/tmp"
)

In [ ]:
df = pd.read_csv("/tmp/customer_shopping_behavior.csv")

display(df.head())

In [ ]:
df.info()

In [ ]:
# Summary Statistics using .describe()
df.describe(include='all')

In [ ]:
df.isnull().sum()

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform( lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
df.columns= df.columns.str.lower()
df.columns= df.columns.str.replace(' ','_')
df = df.rename( columns={ 'purchase_amount_(usd)': 'purchase_amount'})

In [ ]:
df.columns

In [ ]:
labels= ['Young_Adult', 'Adult', 'Middle_aged', 'Senior']
df['age_group'] = pd.qcut( df['age'], q=4, labels=labels)
df[['age','age_group']].head(10)

In [ ]:
df.head()

In [ ]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

In [ ]:
df[['discount_applied','promo_code_used']].head(10)

In [ ]:
df = df.drop('promo_code_used', axis=1)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
%%sql -r dataframe_2
CREATE SCHEMA IF NOT EXISTS MINOR_PROJECT.CLEAN;

In [ ]:

# Save cleaned Pandas DataFrame into Snowflake

from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Write cleaned data into CLEAN schema
result = session.write_pandas(
    df,
    table_name="CUSTOMER_SHOPPING_BEHAVIOR",
    database="MINOR_PROJECT",
    schema="CLEAN",
    auto_create_table=True,
    overwrite=True,
    quote_identifiers=False
)

print("Cleaned business table created successfully!")
print(result)
